In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Earth AI Remote Sensing: DIOR Object Detection

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/notebooks/deploy-notebook?download_url=https://github.com/google-research/remote-sensing/raw/refs/heads/main/remote_sensing/models/notebooks/Segmentation_Best_Practice_Example.ipynb">
      <img alt="Workbench logo" src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" width="32px"><br> Run in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw%2Egithubusercontent%2Ecom%2Fgoogle%2Dresearch%2Fremote%2Dsensing%2Fmaster%2Fremote%5Fsensing%2Fmodels%2Fnotebooks%2FSegmentation%5FBest%5FPractice%5FExample%2Eipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/google-research/remote-sensing/blob/main/remote_sensing/models/notebooks/Segmentation_Best_Practice_Example.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
  </td>
</tr></tbody></table>

# 🛰️ Earth AI Remote Sensing: DLRSD Training Example

This notebook provides an end-to-end semantic segmentation example on the
**DLRSD (Dense Labeling Remote Sensing Dataset)** using the `Earth AI Remote
Sensing` library. It demonstrates how to load a pretrained remote sensing Vision
Transformer encoder+decoder, attach a per-pixel classifier to it,
fine-tune the full model, and evaluate segmentation performance across 17
land-cover classes. All the experiments in this notebook were conducted on a
single H100 GPU with 40GB of VRAM.

--------------------------------------------------------------------------------

### 🛠️ Core Capabilities Demonstrated

*   **Pretrained Encoder+Decoder Usage:** Loading a pretrained
    `ViTEncoderDecoderModel` checkpoint and building a
    `ViTEncoderDecoderSegmentation` module on top of it, to act as the
    segmentation model.

*   **Augmentations:** Training the model using a unified segmentation
    augmentations and processing library.

*   **Loss Function:** Training the model with a progressive weighted
    `CombinedLoss` consisting of `SegmentationCrossEntropyLoss` and
    `SegmentationJaccardLoss`.

*  **Optimization:** Training the model using different optimizers and
    schedulers for the different components of the model.

*  **Metrics and Visualizations:** Tracking losses, segmentation metrics, and
    learning rates, as well as visualization of segmentation examples.

--------------------------------------------------------------------------------

### 📦 Library Modules Overview

| Module | Primary Functionality |
| ------ | --------------------- |
| `remote_sensing.models.augmentations` | Provides preprocessing utilities for segmentation labels, weights, normalization, and validation  checks. |
| `remote_sensing.models.dense_prediction` | Defines pretrained encoder-decoder segmentation models. |
| `remote_sensing.models.losses` | Provides segmentation losses such as Dice loss, Jaccard loss, Lovasz-Softmax loss, Focal loss, and combined  losses. |
| `remote_sensing.models.metrics` | Provides segmentation metrics tracking over multiple epochs. |
| `remote_sensing.models.segmentation_datasets` | Provides a full DataLoader for training and evaluation of segmentation datasets. |
| `remote_sensing.models.visualizations` | Provides visualization utilities for segmentation outputs. |


# ⚙️ Setup

The next section installs the Remote Sensing code.
The code depends on HuggingFace transformers, and requires version 5.5.

In [ ]:
# @title Installing the Remote Sensing code

!pip install git+https://github.com/google-research/remote-sensing.git



In [ ]:
# @title Imports

import io
import itertools
import os
import zipfile

from IPython import display
from matplotlib import patches as mpatches
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import PIL
from remote_sensing.models import dense_prediction
from remote_sensing.models import losses
from remote_sensing.models import metrics
from remote_sensing.models import segmentation_dataset
from remote_sensing.models import visualizations
from sklearn import metrics as sklearn_metrics
from sklearn.model_selection import train_test_split
import torch
import torchvision
import tqdm

# 📂 Dataset Preparation

## ❓ About the dataset

The DLRSD dataset is a 17-class dataset for remote sensing image segmentation,
containing 2100 images of size 256x256, canonically split to 80% for training
and 20% for testing.

The 17 classes are:

id  | name
--- | -----------
1   | Airplane
2   | Bare Soil
3   | Building
4   | Car
5   | Chaparral
6   | Court
7   | Dock
8   | Field
9   | Grass
10  | Mobile Home
11  | Pavement
12  | Sand
13  | Sea
14  | Ship
15  | Tanks
16  | Trees
17  | Water

More information can be found
[here](https://sites.google.com/corp/view/zhouwx/dataset?pli=1#h.p_hQS2jYeaFpV0).

## 🏗️ Dataset preparation

For processing the pipeline, the code:

-   Downloads a copy of the ZIP file locally.
-   Parses pairs of RGB TIFF image, and a label PNG image.
-   Converts the pairs to a dictionary of:

    `{"image": FloatTensor[3, 256, 256], "label": LongTensor[256, 256]}`.

    The `image` tensor contains RGB values between 0 and 1. The `label` tensor
    contains numbers between 0 and 16 (originally the label values are between 1
    and 17, but the code subtracts 1, in order to be compatible with a one-hot
    encoding that will happen later).

-   Finally, splits the dataset randomly (with a constant seed) to `train` (80%)
    and `test` (20%), according to a the benchmark's standard split.

In [ ]:
# @title Downloading the dataset

# Go to https://www.kaggle.com/datasets/asjad2024/dlrsd
# and download the DLRSD.zip file to GCS or to the VM.
DATASET_PATH = "gs://path/to/datasets/dlrsd/DLRSD.zip"


os.makedirs("./dataset", exist_ok=True)

local_dataset_path = os.path.join("./dataset", "DLRSD.zip")

if not os.path.exists(local_dataset_path):
    print("Copying dataset from GCS")
    !gcloud storage cp {DATASET_PATH} {local_dataset_path}
else:
    print(f"Dataset already exists at {local_dataset_path}")

In [ ]:
# @title Parsing the dataset

all_samples = []

decode_image = lambda x: torchvision.transforms.functional.to_tensor(
    PIL.Image.open(io.BytesIO(x))
)
decode_label = lambda x: torch.LongTensor(
    np.array(PIL.Image.open(io.BytesIO(x))) - 1  # Convert to zero-based.
)

print("Reading zip file")
with open(local_dataset_path, "rb") as f:
  with zipfile.ZipFile(f, "r") as zip_ref:
    zip_file_list = zip_ref.namelist()
    zip_file_list = [
        i
        for i in zip_file_list
        if i.startswith("DLRSD/Images/") and i.endswith(".tif")
    ]
    zip_file_list.sort()  # For consistency
    for image_fn in tqdm.tqdm(zip_file_list):
      label_fn = image_fn.replace("Images", "Labels").replace("tif", "png")
      image = decode_image(zip_ref.open(image_fn).read())
      label = decode_label(zip_ref.open(label_fn).read())
      all_samples.append({"image": image, "label": label})

print("\nTotal number of images:", len(all_samples))
assert len(all_samples) == 2100

train, test = train_test_split(all_samples, test_size=0.2, random_state=42)

del all_samples

print("Train: ", len(train))
print("Test: ", len(test))
assert len(train) == 1680
assert len(test) == 420

In [ ]:
# @title Defining dataset-specific constants

CLASS_NAMES = {
    0: "Airplane",
    1: "Bare Soil",
    2: "Building",
    3: "Car",
    4: "Chaparral",
    5: "Court",
    6: "Dock",
    7: "Field",
    8: "Grass",
    9: "Mobile Home",
    10: "Pavement",
    11: "Sand",
    12: "Sea",
    13: "Ship",
    14: "Tanks",
    15: "Trees",
    16: "Water",
}

NUM_CLASSES = len(CLASS_NAMES)
DATASET_IMAGE_SIZE = 256

## 🤖 Model Preparation

This section loads a pretrained Remote Sensing Encoder-Decoder model, which can
be finetuned for semantic segmentation tasks.

The model consists of a ViT-L/16 encoder, and a ViT-lite decoder.

The encoder takes any image size (with height and width divisible by 16, and RGB
channels), and encodes it to (H/16 x W/16) tokens, each of dimension 1024.

The decoder then decodes these tokens further, then, using Depth-to-Space,
outputs per-pixel features.

Finally, the code here attaches a new projection head, from the per-pixel
features to the per-pixel logits.

In [ ]:
# @title Download model

MODEL_NAME = "MODEL_NAME"
MODEL_PATH = f"gs://path/to/models/{MODEL_NAME}"



os.makedirs("./models", exist_ok=True)
local_model_path = os.path.join("./models", MODEL_NAME)

if not os.path.exists(local_model_path):
    print("Copying model from GCS")
    os.makedirs(local_model_path, exist_ok=True)
    !gcloud storage cp -R {MODEL_PATH}/* {local_model_path}
else:
    print(f"Model already exists at {local_model_path}")

In [ ]:
# @title Initialize model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {torch.cuda.device_count()} device(s) of type `{device}`")

geofm = dense_prediction.ViTEncoderDecoderModel.from_pretrained(
    local_model_path
)
model = dense_prediction.ViTEncoderDecoderSegmentation(geofm, NUM_CLASSES).to(
    device
)

# 🧙 Recommended best practices

A few best practices are recommended for finetuning a pretrained segmentation model.

## ⏱️ Separate learning rates and schedulers for different model components:
   
   The model consists of a pretrained Encoder, pretrained Decoder, and a new
   projection layer. It's recommended to tune the encoder with a much lower
   learning rate than the decoder and the projection layer. Furthermore, it's
   recommended to freeze the encoder for a few epochs, before unfreezing it and
   tuning it like the decoder, to allow the decoder and the projection layer to
   stabilize first.
   In order to make it easier to use separate schedulers, a separate optimizer
   for each sub component is created as well.

   Notice: The separation of components defined here is specific for the
   `VitEncoderDecoderSegmentation` architecture. For other architectures, it's
   recommeneded to separate the encoder from the remaining parts as well, but
   the implementation may be different.

## 🎯 Progressive combined loss:

There are multiple useful losses for semantic segmentation, including Cross
Entropy and Jaccard Loss. Cross Entropy is a fast and efficient learner, but
does not optimize for mean-IoU, while Jaccard Loss learns more slowly, but
optimizes mean-IoU by its definition. In order to best utilize both, a
`ProgressiveCombinedLoss` object is provded by the `losses` library. It starts
by giving each of these two losses a weight of 50%, however, after several
epochs, it starts increasing the weight of the Jaccard Loss until reaching 100%
of the weight, and 0% for the Cross Entropy loss. At this point, the Cross
Entropy loss may start to increase, while the Jaccard loss continues to
decrease, leading to higher IoU.

In [ ]:
# @title Helper functions for good practices

def vit_encoder_decoder_optimizers(
    segmentation_model: dense_prediction.ViTEncoderDecoderSegmentation,
    encoder_lr: float = 1e-5,
    decoder_lr: float = 1e-3,
    weight_decay: float = 1e-4,
) -> dict[str, torch.optim.Optimizer]:
  """Creates different optimizers for the Encoder and for the rest of the model.

  Both optimizers use AdamW, with the same weight decay, only a different set of
  parameters, and a different learning rate.

  Args:
    encoder_lr: The learning rate for the encoder.
    decoder_lr: The learning rate for the decoder.
    weight_decay: The weight decay for both.

  Returns:
    A dictionary of {
      "Encoder": torch.optim.Optimizer,
      "Decoder": torch.optim.Optimizer,
    }
  """
  optimizers = {
      "Encoder": torch.optim.AdamW(
          segmentation_model.encoder_decoder.encoder.parameters(),
          lr=encoder_lr,
          weight_decay=weight_decay,
      ),
      "Decoder": torch.optim.AdamW(
          itertools.chain(
              segmentation_model.encoder_decoder.decoder.parameters(),
              segmentation_model.encoder_decoder.norm.parameters(),
              segmentation_model.projection.parameters(),
          ),
          lr=decoder_lr,
          weight_decay=weight_decay,
      ),
  }

  total_params = 0
  for name, optimizer in optimizers.items():
    num_params = sum(p.numel() for p in optimizer.param_groups[0]["params"])
    total_params += num_params
    print(f"Optimizer{name}: {num_params} params")

  print(f"Total: {total_params}")
  assert total_params == sum(p.numel() for p in segmentation_model.parameters())

  return optimizers


def finetuning_schedulers(
    optimizers: dict[str, torch.optim.Optimizer],
    total_iters: int = 60,
    stage1_iters: int | None = None,
    stage2_iters: int | None = None,
) -> dict[str, torch.optim.lr_scheduler.LRScheduler]:
  """Creates different schedulers for the encoder and for the decoder.

  For each component, it creates a 3-stages scheduler:

  Stage one: The encoder is frozen, while the decoder is warmed up.
  Stage two: The encoder is warmed up, while the decoder is kept with a constant
    learning rate.
  Stage three: The encoder and the decoder are cooled down.

  By default, stage one and stage two are 10% of the iterations each, and stage
  three is the remaining 80%.

  Note that these schedulers can either work at an epoch level, or a train step
  level.

  Args:
    optimizers: A dictionary of {
      "Encoder": torch.optim.Optimizer,
      "Decoder": torch.optim.Optimizer,
    }
    total_iters: The total number of training iterations.
    stage1_iters: The number of iterations for stage one, or None to take the
      default 10% of the total iterations.
    stage2_iters: The number of iterations for stage two, or None to take the
      default 10% of the total iterations.

  Returns:
    A dictionary of {
      "Encoder": torch.optim.lr_scheduler.LRScheduler,
      "Decoder": torch.optim.lr_scheduler.LRScheduler,
    }
  """

  # Calculate the number of iterations for each stage.
  if stage1_iters is None:
    stage1_iters = total_iters // 10
  if stage2_iters is None:
    stage2_iters = total_iters // 10
  stage3_iters = total_iters - stage1_iters - stage2_iters

  # Initialize the schedulers for each stage.
  encoder_stages = []
  decoder_stages = []
  milestones = []

  # Stage one: freeze the encoder, warm up the decoder.
  if stage1_iters > 0:
    encoder_stages.append(
        torch.optim.lr_scheduler.ConstantLR(
            optimizers["Encoder"],
            factor=0,
            total_iters=stage1_iters,
        )
    )
    decoder_stages.append(
        torch.optim.lr_scheduler.LinearLR(
            optimizers["Decoder"],
            start_factor=0.1,
            end_factor=1,
            total_iters=stage1_iters,
        )
    )
    milestones.append(stage1_iters)

  # Stage two: warm up the encoder, keep the decoder warm.
  if stage2_iters > 0:
    encoder_stages.append(
        torch.optim.lr_scheduler.LinearLR(
            optimizers["Encoder"],
            start_factor=0.1,
            end_factor=1,
            total_iters=stage2_iters,
        )
    )
    decoder_stages.append(
        torch.optim.lr_scheduler.ConstantLR(
            optimizers["Decoder"],
            factor=1,
            total_iters=stage2_iters,
        )
    )
    milestones.append(stage1_iters + stage2_iters)

  # Stage three: cool down the encoder and the decoder.
  if stage3_iters > 0:
    encoder_stages.append(
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizers["Encoder"],
            T_max=stage3_iters,
        )
    )
    decoder_stages.append(
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizers["Decoder"],
            T_max=stage3_iters,
        )
    )
    milestones.append(total_iters)

  # Return the schedulers, each one a SequentialLR object.
  return {
      "Encoder": torch.optim.lr_scheduler.SequentialLR(
          optimizers["Encoder"],
          encoder_stages,
          milestones[:-1],
      ),
      "Decoder": torch.optim.lr_scheduler.SequentialLR(
          optimizers["Decoder"],
          decoder_stages,
          milestones[:-1],
      ),
  }


def combined_segmentation_loss(
    total_iters: int,
    jaccard_start_weight: float = 0.5,
    jaccard_end_weight: float = 1.0,
    shift_start_fraction: float = 0.5,
    shift_end_fraction: float = 0.8,
) -> losses.ProgressiveCombinedLoss:
  """A progressive combined loss for segmentation.

  Combines Cross Entropy loss and Jaccard loss, with weights progressing over
  time.
  The process starts with `jaccard_start_weight` weight for Jaccard and
  `1-jaccard_start_weight` for Cross Entropy. After `shift_start_fraction` of
  the training, the weights starts shifting towards Jaccard loss. By
  `shift_end_fraction` of the training, the weights are fixed to
  `jaccard_end_weight` for Jaccard and `1-jaccard_end_weight` for Cross Entropy.

  Args:
    total_iters: The total number of training iterations.
    jaccard_start_weight: The starting weight for the Jaccard loss.
    jaccard_end_weight: The ending weight for the Jaccard loss.
    shift_start_fraction: The fraction of training iterations after which the
      weights start shifting.
    shift_end_fraction: The fraction of training iterations after which the
      weights are fixed.

  Returns:
    A ProgressiveCombinedLoss object.
  """

  ls_fns = [
      losses.SegmentationCrossEntropyLoss(),
      losses.SegmentationJaccardLoss(),
  ]

  def weights_provider(cur_iter: int, total_iters: int) -> list[float]:
    cur_fraction = cur_iter / total_iters
    if cur_fraction < shift_start_fraction:
      weight = jaccard_start_weight
    elif cur_fraction > shift_end_fraction:
      weight = jaccard_end_weight
    else:
      weight = jaccard_start_weight + (jaccard_end_weight - jaccard_start_weight) * (
          cur_fraction - shift_start_fraction
      ) / (shift_end_fraction - shift_start_fraction)

    return [1 - weight, weight]

  return losses.ProgressiveCombinedLoss(ls_fns, weights_provider, total_iters)

# 🎨 Data augmentation and processing

Data augmentation is a critical feature for reducing overfitting to the train
dataset. It is especially important for small datasets, such as DLRSD.

The notebook uses the provided `segmentation_dataset` library for applying all
the sample-level augmentations and processing, as well as batch-level
augmentations and processing.

The main approach is to apply spatial augmentations on the images, labels, and
weights, simultaneously. To do that, the label is first one-hot encoded, and
concatenated to the RGB channels, along with an extra 1-initialized weight
tensor. This results in a `(C + D + 1) x H x W` shaped tensor, where `C` is the
number of image channels (3 for RGB images), and `D` is the number of classes
(17 for DLRSD).

The following augmentations on these stacked tensors (in the training dataset
only):
-   Random Crop and Resize:
    
    DLRSD images are 256x256, however, the model can work with different image
    sizes (although 256x256 is used anyway).
    The samples are randomly cropped and resized to the expected image size.
    The scale range of the cropped image within the full image is configurable.

    If random resize is not applied, the samples are deterministically resized
    to the model expected size.

-   Random Flipping:

    Randomly flips the image+label+weight horizontally, in probability 50%.
    Note that vertical flip and diagonal flip are also possible, but will not
    add any more entropy if Random Rotation is applied.

-   Random Rotation:

    Randomly rotates the image+label+weight around the center. The corners are
    padded with zeros, leading to black pixels, undefined labels, and zero
    weights. The angle range is configurable.

-   CutMix and MixUp:

    Randomly cuts and mixes images+labels+weights within a single batch. The
    alpha parameter for CutMix and for MixUp is configurable.

After all the augmentations are applied, the tensors are split back by taking
the first `C` channels as an Image, the next `D` as soft labels, and the last
channel as a weight.

In addition, the image is normalized using ImageNet RGB Mean/STD values.

For training, we recommend applying many augmentations.

For evaluation, we recommend not applying any augmentation.

In [ ]:
# @title Dataset Hyperparemeters

TRAIN_CONFIG = segmentation_dataset.DatasetConfig(
    batch_size=4,
    image_size=256,
    num_classes=NUM_CLASSES,
    shuffle=True,
    random_resize_probability=1.0,
    resize_scale_range=(0.6, 1.0),
    random_flip_probability=0.5,
    random_rotate_probability=1.0,
    rotation_range=(-180.0, 180.0),
    random_cutmix=False,
)

# Note: shuffle=True is used, meaning that on each epoch the test dataset is
# evaluated using a different permuatation. This will not change the metrics,
# but allows the visualizations to show different random samples on each epoch.
TEST_CONFIG = segmentation_dataset.DatasetConfig(
    batch_size=8,
    image_size=256,
    num_classes=NUM_CLASSES,
    shuffle=True,
)

train_loader = segmentation_dataset.create_data_loader(train, TRAIN_CONFIG)
test_loader = segmentation_dataset.create_data_loader(test, TEST_CONFIG)

# 🔧 Training hyperparameters

There are many different hyperparameters that can help improve the learning of
the model. Here, a small set of commonly used parameters is provided.

In [ ]:
# @title Training Hyperparameters

# The number of training epochs. Defaults to `60` which is useful for demo
# purposes, though different values may be optimal, depending on the size of the
# dataset.
NUM_EPOCHS = 60

# As described earlier, these are the learning rates used for the different
# parts of the model.
ENCODER_LR = 1e-5
DECODER_LR = 1e-3

# The AdamW gradient-decoupled weight decay parameter.
WEIGHT_DECAY = 1e-4

# Clipping of the gradient norm, to prevent catastrophic forgetting in case of a
# very extreme batch.
GRAD_NORM_CLIP = 10.0

# Whether to enable auto-casting and grad scaling, and tune the model with
# float16.
ENABLE_AUTOCAST = True

optimizers = vit_encoder_decoder_optimizers(
    model, ENCODER_LR, DECODER_LR, WEIGHT_DECAY
)
schedulers = finetuning_schedulers(optimizers, NUM_EPOCHS)
criterion = combined_segmentation_loss(NUM_EPOCHS)
scaler = torch.amp.GradScaler(enabled=ENABLE_AUTOCAST)

# 🏎️ Training Loop, Evaluation & Monitoring

This section implements the main training loop, metric tracking, and visualization.

## 🎯 Training loop

The training loop is relatively straight-forward, with a few minor
modifications:

- Multiple optimizers are used, for the different components of the model.

- An optional Grad Scaler is used, for faster training using float16.

- Statistics about the performance on the train and the test datasets are
  accumulated on each epoch using the `segmentation_metrics_tracker` object.

- Multiple LR Schedulers are updated each epoch, as well as the
  `ProgressiveCombinedLoss` object.

- Metrics tracked by the `segmentation_metrics_tracker` are plotted, along
  with visualizations of random test dataset images, using the `visualizations`
  library.

## 📈 Metric tracking

A helper class is provided by the `metrics` library for accumulating
segmentation metrics within a full epoch. This is done by running forward pass
of the model on multiple batches, accumulating the confusion matrix between the
predictions and the labels, and then calculating all the metrics from the
confusion matrix.

## 📊 Visualizations

The metrics tracking library collects a large `DataFrame` object, consisting of
per-epoch metrics. The table can then be queried and plotted, including
line-graphs for metrics such as losses, per-class IOUs, and overall mean IOU.
The latest confusion matrix is plotted as well. In addition, the learning rates
at each epoch are tracked and visulaized. Finally, a random image is chosen from
the test dataset, and visualized as a 4-grid image, consisting of the image, the
label, the prediction, and the prediction correctness.

In [ ]:
# @title Training Loop

# Helper objects
metrics_tracker = metrics.segmentation_metrics_tracker(
    CLASS_NAMES, {
        "Dice": losses.SegmentationDiceLoss(),
        "Lovasz": losses.SegmentationLovaszSoftmaxLoss(),
        "Xent": losses.SegmentationCrossEntropyLoss(),
        "Jaccard": losses.SegmentationJaccardLoss(),
        "Combined": criterion,
    }
)
lr_tracker = pd.DataFrame(columns=list(optimizers.keys()))
vis_config = visualizations.SegmentationVisualizationConfig.auto(NUM_CLASSES)
denormalize = visualizations.Denormalize(
    torch.tensor(TRAIN_CONFIG.normalization_mean),
    torch.tensor(TRAIN_CONFIG.normalization_std),
)

# Main loop
for epoch in range(NUM_EPOCHS):

  # Training
  model.train()

  for sample in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
    images = sample["image"].to(device)
    labels = sample["label"].to(device)
    weights = sample["weight"].to(device)

    # Zero gradients for all optimizers.
    for optimizer in optimizers.values():
      optimizer.zero_grad()

    # Forward pass for the model and the combined loss.
    with torch.autocast(device_type="cuda", enabled=ENABLE_AUTOCAST):
      logits = model(images)
      loss = criterion(logits, labels, weights)

    # Backward pass and optimizer step.
    scaler.scale(loss).backward()
    for optimizer in optimizers.values():
      scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_NORM_CLIP)
    for optimizer in optimizers.values():
      scaler.step(optimizer)
    scaler.update()

    # Accumulate metrics on the train dataset.
    metrics_tracker.add_batch("Train", logits, labels, weights)

  # Validation
  model.eval()

  with torch.no_grad():
    for d in tqdm.tqdm(test_loader, desc=f"Epoch {epoch} [Val]"):
      images = d["image"].to(device)
      labels = d["label"].to(device)
      weights = d["weight"].to(device)

      # Forward pass for the model.
      with torch.autocast(device_type="cuda", enabled=ENABLE_AUTOCAST):
        logits = model(images)

      # Accumulate metrics on the test dataset.
      metrics_tracker.add_batch("Val", logits, labels, weights)

  # Update metrics tracker and learning rate tracker.
  metrics_tracker.next_epoch()
  lr_tracker.loc[epoch] = {
      "Encoder": schedulers["Encoder"].get_last_lr()[0] / ENCODER_LR,
      "Decoder": schedulers["Decoder"].get_last_lr()[0] / DECODER_LR,
  }

  # Step the learning rate schedulers and the progressive combined loss.
  for scheduler in schedulers.values():
    scheduler.step()
  criterion.step()

  # Visualization
  fig = plt.figure(figsize=(24, 12), facecolor="white")
  gs = fig.add_gridspec(2, 4)

  # Plot losses.
  ax_loss = fig.add_subplot(gs[0, 0])
  metrics_tracker.filter_and_rename(
      regexp_from="(.*)_loss_(.*)", regexp_to="\\0 - \\1"
  ).plot(ax=ax_loss, lw=2, marker="o", markersize=3)
  ax_loss.set_title("Losses", fontsize=14)
  ax_loss.grid(alpha=0.2)
  ax_loss.set_facecolor("white")

  # Plot per-class and total IOU.
  ax_classes = fig.add_subplot(gs[0, 1])
  metrics_tracker.filter_and_rename(
      regexp_from="Val_class_(.*)_iou", regexp_to="\\0"
  ).plot(
      ax=ax_classes,
      lw=1,
      marker="o",
      markersize=3,
      color=vis_config.label_palette.numpy().tolist(),
  )
  metrics_tracker.filter_and_rename(
      regexp_from="(.*)_mean_iou", regexp_to="\\0 - Mean IOU"
  ).plot(ax=ax_classes, lw=4, marker="o", markersize=4)
  ax_classes.set_title("Val Class IOU + Train Mean IOU", fontsize=14)
  ax_classes.grid(alpha=0.2)
  ax_classes.set_facecolor("white")

  # Plot learning rates.
  ax_lrs = fig.add_subplot(gs[0, 2])
  lr_tracker.plot(ax=ax_lrs, lw=2, marker="o", markersize=3)
  ax_lrs.set_title("Learning Rate Schedulers", fontsize=14)
  ax_lrs.grid(alpha=0.2)
  ax_lrs.set_facecolor("white")

  # Plot confusion matrix.
  ax_confusion = fig.add_subplot(gs[0, 3])
  confusion_matrix = metrics_tracker.last()["Val_confusion_matrix"]
  confusion_matrix /= confusion_matrix.sum(axis=1)[:, None]  # Normalize on 'gt'
  confusion_matrix_display = sklearn_metrics.ConfusionMatrixDisplay(
      confusion_matrix, display_labels=CLASS_NAMES.values()
  )
  confusion_matrix_display.plot(
      ax=ax_confusion,
      include_values=False,
      xticks_rotation="vertical",
      colorbar=False,
  )
  ax_confusion.set_title("Val Confusion Matrix", fontsize=14)

  # Visualize a random validation sample.
  sample = next(iter(test_loader))
  sample_image = sample["image"][:1].to(device)
  sample_label = sample["label"][:1].cpu()
  sample_weight = sample["weight"][:1].cpu()
  with torch.no_grad():
    prediction = torch.nn.functional.softmax(model(sample_image), dim=1).cpu()
  denorm_image = denormalize(sample_image).cpu()
  vis_img = visualizations.visualize_segmentation(
      denorm_image, sample_label, prediction, sample_weight, vis_config
  ).permute(1, 2, 0)

  ax_img = fig.add_subplot(gs[1, 0:4])
  ax_img.imshow(vis_img)
  ax_img.set_title(f"Image / Label / Prediction / Correctness")
  ax_img.axis("off")
  ax_img.set_facecolor("white")

  legend_patches = [
      mpatches.Patch(
          color=tuple(vis_config.label_palette[c].numpy()) + (1,),
          label=CLASS_NAMES[c],
      )
      for c in CLASS_NAMES
  ]
  ax_img.legend(
      handles=legend_patches,
      bbox_to_anchor=(1.05, 1),
      loc="upper left",
      facecolor="white",
      edgecolor="black",
      labelcolor="black",
  )

  # Rerender the plots.
  display.clear_output(wait=True)
  plt.tight_layout()
  plt.show()